# 19.07 - Video Dataset

**Notebook type:** Practice notebook with theory, exercises, TODO cells, smoke checks, and test cases.

**Daily output:** A `VideoDataset` skeleton that returns one clip as a CPU `torch.float32` tensor with shape `[T, C, H, W]` and its label, plus a `DataLoader` batch with shape `[B, T, C, H, W]`.

Today turns the video-reading utility into a reusable training input pipeline. The emphasis is on explicit tensor contracts, deterministic temporal sampling, stable label IDs, bounded caching, and memory-aware batching.

## Core Ideas

### 1. Keep the temporal dimension visible

An image tensor usually has layout `[C, H, W]`. A video clip adds time: `[T, C, H, W]`. A `DataLoader` adds the batch dimension: `[B, T, C, H, W]`. Keeping this convention explicit prevents a model from silently treating time as channels or batch items.

### 2. Decode first, then sample valid frames

Container metadata can overstate the number of decodable frames. For a small, robust baseline, decode valid frames sequentially and create uniform indices from the decoded count. Repeated indices make a short video produce the same fixed `T` as a long video.

### 3. Convert layout, color, and dtype once

OpenCV decodes BGR `uint8` arrays in `[H, W, C]`. The dataset boundary converts them to RGB, resizes them, changes the layout to `[T, C, H, W]`, and normalizes to `float32` in `[0, 1]`. Downstream models then receive one consistent contract.

### 4. Freeze the label mapping

Text labels must map to integer class IDs deterministically. This notebook sorts the unique label names, so every row, batch, checkpoint, and prediction uses the same `class_to_idx` mapping. In a real project, save that mapping with the model checkpoint.

### 5. Cache with a budget

Caching decoded clips saves repeated video I/O but consumes memory. A float32 clip uses `T * C * H * W * 4` bytes. The dataset below uses a bounded least-recently-used cache; `cache_size=0` disables it. With multiple `DataLoader` workers, each worker process owns a separate dataset and therefore a separate cache, so total memory can multiply.

### 6. Fail loudly on bad inputs

A missing file, empty decode, invalid clip length, or unknown manifest schema should raise a clear error during development. Returning fabricated zero clips can hide data problems and contaminate validation results.

In [ ]:
import os
import csv
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

SEED = 19
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "_day19_video_data"
VIDEO_DIR = os.path.join(DATA_DIR, "videos")
MANIFEST_PATH = os.path.join(DATA_DIR, "labels.csv")
CLIP_LENGTH = 8
FRAME_SIZE = (40, 40)  # (height, width)


## Prepared Video Data

The provided cell creates six tiny deterministic MJPG/AVI videos and a `labels.csv` manifest. Red-dominant clips represent `move_left`; blue-dominant clips represent `move_right`. Clip lengths vary, so fixed-length sampling must both repeat and discard frames. This setup is provided and is not a learner TODO.

**Return structure — `make_day19_video_data(output_dir)`:** a `list[dict]` of length 6. Every dictionary has exactly these keys:

- `video_path`: Python `str` containing the generated video path, relative to the notebook working directory.
- `label`: Python `str`, either `"move_left"` or `"move_right"`.
- `frame_count`: positive Python `int` giving the number of frames written.

The function also creates `output_dir/videos/*.avi` and `output_dir/labels.csv`. The CSV has columns `video_path,label`, with paths relative to the CSV's directory.

In [ ]:
def make_day19_video_data(output_dir=DATA_DIR):
    video_dir = os.path.join(output_dir, "videos")
    os.makedirs(video_dir, exist_ok=True)

    specs = [
        ("move_left", 5),
        ("move_right", 7),
        ("move_left", 9),
        ("move_right", 11),
        ("move_left", 13),
        ("move_right", 15),
    ]
    height, width = 48, 64
    fps = 8.0
    fourcc = cv2.VideoWriter_fourcc(*"MJPG")
    records = []

    for video_id, (label, frame_count) in enumerate(specs):
        filename = f"clip_{video_id:02d}_{label}.avi"
        video_path = os.path.join(video_dir, filename)
        writer = cv2.VideoWriter(video_path, fourcc, fps, (width, height))
        if not writer.isOpened():
            raise RuntimeError("Could not create demo video: " + video_path)

        for frame_id in range(frame_count):
            frame = np.zeros((height, width, 3), dtype=np.uint8)
            frame[:, :, 1] = np.uint8(35 + video_id * 3)
            if label == "move_left":
                frame[:, :, 2] = np.uint8(170)  # red after BGR -> RGB
                x_center = width - 6 - int((width - 12) * frame_id / max(frame_count - 1, 1))
            else:
                frame[:, :, 0] = np.uint8(170)  # blue after BGR -> RGB
                x_center = 6 + int((width - 12) * frame_id / max(frame_count - 1, 1))
            cv2.rectangle(
                frame,
                (max(0, x_center - 3), 15),
                (min(width - 1, x_center + 3), 32),
                (245, 245, 245),
                thickness=-1,
            )
            writer.write(frame)
        writer.release()

        if not os.path.isfile(video_path) or os.path.getsize(video_path) == 0:
            raise RuntimeError("Demo video was not written: " + video_path)
        records.append(
            {"video_path": video_path, "label": label, "frame_count": int(frame_count)}
        )

    manifest_path = os.path.join(output_dir, "labels.csv")
    with open(manifest_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["video_path", "label"])
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "video_path": os.path.join("videos", os.path.basename(record["video_path"])),
                    "label": record["label"],
                }
            )

    return records


video_records = make_day19_video_data()
print("prepared videos:", len(video_records))
print("manifest:", MANIFEST_PATH)
print("frame counts:", [record["frame_count"] for record in video_records])


## Exercise 19-A: Build Uniform Temporal Indices

Implement `uniform_sample_indices(frame_count, clip_length)`. Return exactly `clip_length` monotonically non-decreasing indices spanning the first through last decoded frame. Repetition is correct when `frame_count < clip_length`.

**Return structure — `uniform_sample_indices(frame_count, clip_length)`:** one NumPy `ndarray` with shape `[T]`, dtype `np.int64`, where `T = clip_length`. Values are monotonically non-decreasing integers in `[0, frame_count - 1]`; the first value is `0`, and the last value is `frame_count - 1`.

In [ ]:
# TODO 19-A
def uniform_sample_indices(frame_count, clip_length):
    # Validate both values, create evenly spaced positions, round them,
    # and return exactly clip_length indices with dtype np.int64.
    raise NotImplementedError("Complete Exercise 19-A")


# Smoke check: short videos should repeat some positions.
smoke_indices = uniform_sample_indices(frame_count=5, clip_length=8)
print("sample indices:", smoke_indices.tolist())


## Exercise 19-B: Decode One Video into `[T, C, H, W]`

Implement `load_video_clip(video_path, clip_length=8, frame_size=(40, 40))`. Decode all valid frames sequentially, convert BGR to RGB, resize to `(height, width)`, sample a fixed temporal length, change from channel-last to channel-first layout, and normalize from `uint8` to `float32` in `[0, 1]`. Release `VideoCapture` even if decoding fails.

**Return structure — `load_video_clip(video_path, clip_length, frame_size)`:** one contiguous CPU `torch.Tensor` with shape `[T, C, H, W]`, dtype `torch.float32`, and values in `[0.0, 1.0]`. `T = clip_length`, `C = 3` in RGB order, and `(H, W) = frame_size`.

In [ ]:
# TODO 19-B
def load_video_clip(video_path, clip_length=8, frame_size=(40, 40)):
    # 1. Validate path, clip_length, and (height, width).
    # 2. Decode valid frames sequentially inside try/finally.
    # 3. Convert each frame BGR -> RGB and resize it.
    # 4. Use uniform_sample_indices on the decoded frame count.
    # 5. Stack, permute to [T, C, H, W], make contiguous, and normalize.
    raise NotImplementedError("Complete Exercise 19-B")


# Smoke check: decode the first prepared clip immediately.
smoke_clip = load_video_clip(
    video_records[0]["video_path"], clip_length=CLIP_LENGTH, frame_size=FRAME_SIZE
)
print("clip:", tuple(smoke_clip.shape), smoke_clip.dtype, smoke_clip.device)


## Exercise 19-C: Build a Manifest-Backed `VideoDataset` with Bounded Caching

Implement `VideoDataset`. Read the CSV once in `__init__`, resolve each video path relative to the manifest, create a sorted label mapping, and load clips lazily in `__getitem__`. Maintain a bounded least-recently-used cache. Return clones so a caller cannot mutate the cached tensor in place.

**Return structure — `VideoDataset(manifest_path, clip_length, frame_size, cache_size)`:** one `VideoDataset` instance with:

- `class_to_idx`: `dict[str, int]` mapping sorted unique label names to contiguous IDs `0..C-1`.
- `idx_to_class`: `dict[int, str]`, the exact inverse mapping.
- `__len__()`: Python `int` equal to the number of manifest rows, `N`.
- `__getitem__(index)`: a dictionary with exactly three keys: `clip`, a CPU `torch.float32` tensor `[T, 3, H, W]`; `label`, a scalar CPU `torch.long` tensor; and `video_path`, the resolved Python `str` path.
- `_clip_cache`: `dict[str, torch.Tensor]` containing at most `cache_size` CPU clips; it is empty when `cache_size=0`.

`T = clip_length`, `(H, W) = frame_size`, `N` is the row count, and `C` is the number of unique labels.

In [ ]:
# TODO 19-C
class VideoDataset(Dataset):
    def __init__(self, manifest_path, clip_length=8, frame_size=(40, 40), cache_size=0):
        # Read and validate the manifest, resolve paths, build sorted label maps,
        # and initialize an empty bounded cache plus its LRU order list.
        raise NotImplementedError("Complete VideoDataset.__init__")

    def __len__(self):
        raise NotImplementedError("Complete VideoDataset.__len__")

    def _load_with_cache(self, video_path):
        # On a hit, refresh LRU order and return a clone. On a miss, decode,
        # evict the oldest entry when full, cache a clone, and return the clip.
        raise NotImplementedError("Complete VideoDataset._load_with_cache")

    def __getitem__(self, index):
        # Validate/normalize the index and return the exact three-key sample.
        raise NotImplementedError("Complete VideoDataset.__getitem__")


# Smoke check: construct the dataset and fetch one complete sample.
smoke_dataset = VideoDataset(
    MANIFEST_PATH, clip_length=CLIP_LENGTH, frame_size=FRAME_SIZE, cache_size=2
)
smoke_sample = smoke_dataset[0]
print("dataset length:", len(smoke_dataset))
print("class mapping:", smoke_dataset.class_to_idx)
print("sample:", tuple(smoke_sample["clip"].shape), int(smoke_sample["label"]))


## Exercise 19-D: Estimate the Clip Cache Memory Budget

Implement `estimate_clip_cache_mb(...)`. Use it before choosing a cache size instead of guessing. This estimate covers only raw cached tensor storage; Python objects, decoder buffers, batches, model activations, and per-worker cache copies add more memory.

**Return structure — `estimate_clip_cache_mb(num_clips, clip_length, channels, height, width, dtype_bytes)`:** one non-negative Python `float` containing `num_clips * clip_length * channels * height * width * dtype_bytes / 1024**2` mebibytes.

In [ ]:
# TODO 19-D
def estimate_clip_cache_mb(
    num_clips, clip_length, channels, height, width, dtype_bytes=4
):
    # Validate the requested count and dimensions, compute bytes,
    # then convert bytes to mebibytes using 1024 ** 2.
    raise NotImplementedError("Complete Exercise 19-D")


# Smoke check: estimate this notebook's two-clip float32 cache.
smoke_cache_mb = estimate_clip_cache_mb(
    2, CLIP_LENGTH, 3, FRAME_SIZE[0], FRAME_SIZE[1], dtype_bytes=4
)
print(f"two-clip cache estimate: {smoke_cache_mb:.4f} MiB")


## Exercise 19-E: Batch Video Samples

Implement `make_video_dataloader(dataset, batch_size=2, shuffle=False)`. Use `num_workers=0` for this small notebook so the dataset cache stays in one process and behavior is easy to inspect. A seeded generator makes shuffled order reproducible.

**Return structure — `make_video_dataloader(dataset, batch_size, shuffle)`:** one PyTorch `DataLoader`. Each iteration yields a dictionary with exactly three keys: `clip`, a CPU `torch.float32` tensor `[B, T, 3, H, W]`; `label`, a CPU `torch.long` tensor `[B]`; and `video_path`, a `list[str]` of length `B`. The final batch may have `B < batch_size`.

In [ ]:
# TODO 19-E
def make_video_dataloader(dataset, batch_size=2, shuffle=False):
    # Validate inputs and return a DataLoader with num_workers=0 and
    # a torch.Generator seeded with SEED.
    raise NotImplementedError("Complete Exercise 19-E")


# Smoke check: print the day's required batch shapes.
smoke_loader = make_video_dataloader(smoke_dataset, batch_size=2, shuffle=False)
smoke_batch = next(iter(smoke_loader))
print("batch clips:", tuple(smoke_batch["clip"].shape))
print("batch labels:", tuple(smoke_batch["label"].shape), smoke_batch["label"].tolist())


## Test Cases

Run this cell after completing Exercises 19-A through 19-E. It checks generated files, temporal indices, RGB/layout/dtype/range, label mapping, sample schema, cache bounds and clone safety, memory arithmetic, batch shape, and error handling.

A correct implementation prints exactly `Day 19 tests passed`.

**Return structure — `run_day19_tests()`:** `None`. Success is communicated by completed assertions and the printed confirmation; a failed contract raises `AssertionError` or the expected input-validation exception.

In [ ]:
def run_day19_tests():
    required_names = [
        "uniform_sample_indices",
        "load_video_clip",
        "VideoDataset",
        "estimate_clip_cache_mb",
        "make_video_dataloader",
    ]
    for name in required_names:
        assert name in globals(), "Missing implementation: " + name

    assert os.path.isfile(MANIFEST_PATH)
    assert len(video_records) == 6
    assert all(os.path.isfile(record["video_path"]) for record in video_records)
    assert {record["label"] for record in video_records} == {"move_left", "move_right"}

    short_indices = uniform_sample_indices(5, 8)
    assert short_indices.shape == (8,)
    assert short_indices.dtype == np.int64
    assert short_indices[0] == 0 and short_indices[-1] == 4
    assert np.all(short_indices[1:] >= short_indices[:-1])
    assert len(np.unique(short_indices)) < len(short_indices)

    clip = load_video_clip(video_records[0]["video_path"], 8, (40, 40))
    assert clip.shape == (8, 3, 40, 40)
    assert clip.dtype == torch.float32
    assert clip.device.type == "cpu"
    assert clip.is_contiguous()
    assert float(clip.min()) >= 0.0 and float(clip.max()) <= 1.0
    assert float(clip[:, 0].mean()) > float(clip[:, 2].mean()), "Expected RGB red dominance"

    dataset = VideoDataset(MANIFEST_PATH, 8, (40, 40), cache_size=2)
    assert len(dataset) == 6
    assert dataset.class_to_idx == {"move_left": 0, "move_right": 1}
    assert dataset.idx_to_class == {0: "move_left", 1: "move_right"}
    item0 = dataset[0]
    assert set(item0.keys()) == {"clip", "label", "video_path"}
    assert item0["clip"].shape == (8, 3, 40, 40)
    assert item0["clip"].dtype == torch.float32
    assert item0["label"].shape == torch.Size([])
    assert item0["label"].dtype == torch.long
    assert os.path.isfile(item0["video_path"])

    item0_again = dataset[0]
    assert torch.equal(item0["clip"], item0_again["clip"])
    assert item0["clip"].data_ptr() != item0_again["clip"].data_ptr()
    dataset[1]
    dataset[2]
    assert len(dataset._clip_cache) <= 2
    assert len(dataset._cache_order) <= 2

    no_cache_dataset = VideoDataset(MANIFEST_PATH, 8, (40, 40), cache_size=0)
    no_cache_dataset[0]
    assert no_cache_dataset._clip_cache == {}

    expected_mb = 12 * 8 * 3 * 40 * 40 * 4 / (1024 ** 2)
    actual_mb = estimate_clip_cache_mb(12, 8, 3, 40, 40, 4)
    assert isinstance(actual_mb, float)
    assert abs(actual_mb - expected_mb) < 1e-12

    loader = make_video_dataloader(dataset, batch_size=2, shuffle=False)
    batch = next(iter(loader))
    assert set(batch.keys()) == {"clip", "label", "video_path"}
    assert batch["clip"].shape == (2, 8, 3, 40, 40)
    assert batch["clip"].dtype == torch.float32
    assert batch["label"].shape == (2,)
    assert batch["label"].dtype == torch.long
    assert isinstance(batch["video_path"], list) and len(batch["video_path"]) == 2

    try:
        uniform_sample_indices(0, 8)
        raise AssertionError("frame_count=0 should fail")
    except ValueError:
        pass
    try:
        dataset[len(dataset)]
        raise AssertionError("out-of-range dataset index should fail")
    except IndexError:
        pass
    try:
        load_video_clip("missing_day19_video.avi", 8, (40, 40))
        raise AssertionError("missing video should fail")
    except FileNotFoundError:
        pass

    print("Day 19 tests passed")


run_day19_tests()


## Day 19 Checklist

- [ ] I can explain `[T, C, H, W]` for one clip and `[B, T, C, H, W]` for one batch.
- [ ] I sample from successfully decoded frames and repeat indices for short videos.
- [ ] I convert OpenCV BGR `uint8` frames into contiguous RGB `float32` tensors.
- [ ] I create one deterministic `class_to_idx` mapping and keep it stable.
- [ ] I return a consistent dictionary schema from every dataset item.
- [ ] I can estimate cache memory before selecting `cache_size`.
- [ ] I understand that each multi-worker dataset can own a separate cache.
- [ ] I verified a batch has shape `[B, T, C, H, W]` and labels have dtype `torch.long`.